# 04 — Testing & scenario comparison (Step 4)
Compare **NN vs RF** across the closest-users scenarios — `X ∈ {3,5,10}` × encoding (`pos`/`agg`) plus the
**X=0 no-neighbour baseline** (users sampled at random from the full ACC Arena population) — on
MSE / MAE / R² and training duration, and identify the best configuration.

In [ ]:
# === Project config — Team 8: Throughput Prediction in a Dense 5G deployment (with Transfer Learning) ===
RANDOM_SEED      = 42
RESAMPLE_SECONDS = 120           # time granularity: THE dataset-size knob (professor's guidance: coarsen the
                                 # time grid rather than subsample users). Raw cadence is ~3s with frequent 4s
                                 # steps, duplicate stamps and gaps up to 7s (ACC) / ~1s (Salt&Tar); keep >= 10
                                 # so the nearest-alignment tolerance stays above the worst raw gap.
N_USERS          = None          # ACC Arena users. None = ALL (~12k), so the X-closest neighbourhoods are the
                                 # real ones; an int (e.g. 1500, seeded random sample) biases the neighbourhoods
                                 # (X closest searched within the sample) and is only for quick debug runs.
N_USERS_SALT     = 3000          # Salt&Tar users: ALL of them (only ~1/3 are ever active; activity is id-biased)
X_VALUES         = [3, 5, 10]    # number of closest users to experiment with. X=1 excluded by design:
                                 # heavy co-location makes a single arbitrary neighbour uninformative.
                                 # X=0 (no neighbour features) IS produced and trained: it is the baseline
                                 # that quantifies what the closest-users features actually add.
ENCODINGS        = ["pos", "agg"]# neighbour-feature encodings: "pos" = ordered per-neighbour columns
                                 # (nb0_*, nb1_*, ...), "agg" = order-invariant aggregates over the X
                                 # neighbours (sum/mean/count) — rationale in notebook 02.
BEST_X           = 3             # X used for the transfer-learning experiment (pick the best from notebook 04)
BEST_ENC         = "pos"         # encoding used for the transfer-learning experiment (pick from notebook 04)
OUTLIER_PCT      = 99.0          # drop samples with throughput above this TRAIN percentile (None = keep all).
                                 # EDA (notebook 01): the ~1% extreme samples carry ~2/3 of the total variance.
ACTIVE_ONLY      = True           # regress only on ACTIVE users; idle/off have throughput ~0 by definition
MIN_TRAFFIC_TYPE = 2             # keep traffic_type >= this (2=const_rate, 3=video, 4=gaming, 5=http)

# --- data location ---
DRIVE_FOLDER_ID  = "1s1YCWyhN_Fv5sQraTVs4Rga-ATiKPRgo"   # shared Drive folder containing the zip
ZIP_NAME         = "L5GHDD_Dataset.zip"
DATA_ROOT        = "data/raw"     # dataset folders live here after unzip (local default)
PROCESSED_DIR    = "data/processed"
RESULTS_DIR      = "results"

import os, numpy as np, random
random.seed(RANDOM_SEED); np.random.seed(RANDOM_SEED)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
# === Colab: mount Drive and make outputs PERSIST across notebooks (no-op locally) ===
def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except Exception:
        return False

if in_colab():
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR   = "/content/drive/MyDrive/5g_throughput_team8"   # persistent project folder on your Drive
    PROCESSED_DIR = f"{PROJECT_DIR}/processed"                     # 02 writes here, 03/04/05 read from here
    RESULTS_DIR   = f"{PROJECT_DIR}/results"                       # models, metrics.csv, figures
    print("Outputs persist in:", PROJECT_DIR)
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/figures", exist_ok=True)
os.makedirs(f"{RESULTS_DIR}/models", exist_ok=True)

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
res = pd.read_csv(f"{RESULTS_DIR}/metrics.csv")
if "enc" not in res.columns:                  # metrics.csv from a run before the encoding experiment
    res["enc"] = "pos"
res.sort_values(["model","enc","X"])

## Metrics vs X, per encoding
Solid = positional encoding, dashed = aggregated encoding, dotted horizontal = **X=0 no-neighbour baseline**.
The vertical gap between a curve and its baseline is the *net contribution of the closest-users features*.

In [ ]:
metrics = ["MSE","MAE","R2","train_s"] + (["infer_ms"] if "infer_ms" in res.columns else [])
colors = {"NN": "tab:blue", "RF": "tab:orange"}
styles = {"pos": "-", "agg": "--"}
fig, ax = plt.subplots(1, len(metrics), figsize=(4*len(metrics),4))
for i, metric in enumerate(metrics):
    for model in ["NN","RF"]:
        for enc in [e for e in ["pos","agg"] if e in res.enc.values]:
            d = res[(res.model==model) & (res.enc==enc)].sort_values("X")
            ax[i].plot(d.X, d[metric], marker="o", color=colors[model], linestyle=styles[enc],
                       label=f"{model} {enc}")
        base = res[(res.model==model) & (res.enc=="none")]
        if len(base):                          # X=0 baseline: no neighbour features at all
            ax[i].axhline(base[metric].iloc[0], color=colors[model], linestyle=":", alpha=.7,
                          label=f"{model} X=0")
    ax[i].set_xlabel("X (closest users)"); ax[i].set_title(metric); ax[i].legend(fontsize=8); ax[i].grid(alpha=.3)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_metrics_vs_X.png", dpi=120); plt.show()

## Best configuration and pred-vs-true

In [ ]:
best = res.loc[res.R2.idxmax()]
print("Best:", best[["model","X","enc","MSE","MAE","R2","train_s"]].to_dict())

import numpy as np, joblib
from tensorflow import keras
x, enc = int(best.X), best.enc
sfx = "_agg" if enc == "agg" else ""
d = np.load(f"{PROCESSED_DIR}/acc_X{x}{sfx}.npz"); Xte, yte = d["X_test"], d["y_test"]
if best.model == "NN":
    pred = keras.models.load_model(f"{RESULTS_DIR}/models/nn_X{x}{sfx}.keras").predict(Xte, verbose=0).ravel()
else:
    pred = joblib.load(f"{RESULTS_DIR}/models/rf_X{x}{sfx}.pkl").predict(Xte)

plt.figure(figsize=(5,5))
plt.scatter(yte, pred, s=5, alpha=.3)
lim = [0, max(yte.max(), pred.max())]; plt.plot(lim, lim, "r--")
plt.xlabel("True throughput (Mbps)"); plt.ylabel("Predicted"); plt.title(f"{best.model} (X={x}, {enc})")
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_pred_vs_true.png", dpi=120); plt.show()

## How much do the closest-users features contribute? RF feature importance
Side-by-side for the two encodings of the mid scenario: with the **positional** encoding the co-located
neighbours are interchangeable, so their importance is diluted over many `nbK_*` columns; the **aggregated**
encoding concentrates the same information in a handful of order-invariant columns (`nb_prb_sum` ≈ cell load).

In [ ]:
import json as _json
x_imp = 5                                     # scenario to inspect
fig, axes = plt.subplots(1, 2, figsize=(14,5))
for a, sfx, title in [(axes[0], "", "positional (nbK_*)"), (axes[1], "_agg", "aggregated (nb_*)")]:
    rf = joblib.load(f"{RESULTS_DIR}/models/rf_X{x_imp}{sfx}.pkl")
    cols = _json.load(open(f"{PROCESSED_DIR}/acc_X{x_imp}{sfx}_cols.json"))
    imp = pd.Series(rf.feature_importances_, index=cols).sort_values(ascending=False)
    nb_mask = imp.index.str.startswith("nb")
    print(f"{title:22s} own features {imp[~nb_mask].sum()*100:.1f}%  |  "
          f"neighbour features {imp[nb_mask].sum()*100:.1f}%  (spread over {nb_mask.sum()} columns)")
    top = imp.head(15)[::-1]
    top.plot.barh(ax=a, color=["#e76f51" if c.startswith("nb") else "#2a9d8f" for c in top.index])
    a.set_title(f"RF importance, X={x_imp}, {title}")
fig.suptitle("green = own features, red = neighbour features", y=1.02)
plt.tight_layout(); plt.savefig(f"{RESULTS_DIR}/figures/04_feature_importance.png", dpi=120,
                                bbox_inches="tight"); plt.show()

**Interpretation notes (context from the raw-data analysis).** Users are placed on discrete spots and heavily
co-located (at a given instant, tens of users can share the exact same position), and throughput in this
dataset is strongly **demand-driven**: the user's own `prb` dominates, while neighbour columns share the
remainder. Reading guide:
- **X=0 baseline vs the curves**: the gap is the net value of the closest-users features. If a curve sits on
  the baseline, that scenario's neighbour features add nothing.
- **pos vs agg**: with co-located (tied) neighbours the positional ordering is arbitrary, so `nbK_*` columns
  are permutation noise and the RF dilutes their importance; the aggregates (`nb_prb_sum` ≈ cell load,
  `nb_active_count`) encode the contention mechanism directly and should use fewer, stronger columns.
- **X growth**: check whether R² improves with X or the extra columns only dilute the RF's per-split feature
  sampling (`max_features="sqrt"`), and weigh the training-cost increase.
Users are sampled **at random from the whole venue**, so the neighbour sample is unbiased w.r.t. user ids.

## Takeaways
- Compare how the metrics move across `X ∈ {3,5,10}` **and across encodings (pos vs agg)** for both models,
  always against the **X=0 no-neighbour baseline**.
- Note the **accuracy vs training-cost** trade-off (training duration grows with X / model complexity; the
  aggregated encoding keeps the feature count constant in X).
- Set `BEST_X` **and `BEST_ENC`** in the config of notebook **05** to the best-performing combination for
  the transfer-learning study.